# 3 · Ejercicio 2: isomorfismos (15 min)

Dos tipos son **isomorfos** si puedes ir de uno a otro (`from`) y volver (`to`) sin perder nada. Te piden:
- **a)** demostrarlo con aritmética (contando valores),
- **b)** implementar `from` y `to`.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen, comentadas.

In [1]:
// ─────────────────────────────────────────────────────────────────────────────
// PREÁMBULO — las mismas definiciones que te dan en el examen.
// Ejecuta esta celda antes que ninguna otra del notebook.
// ─────────────────────────────────────────────────────────────────────────────

// Librería de tests. `run(TestX(f))` ejecuta los tests de un apartado.
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.
type Not[P] = P => Nothing
// "P si y solo si Q" = un par de funciones: de P a Q y de Q a P.
type <=>[P, Q] = (P => Q, Q => P)

// Tercio excluso: para cualquier P, o tengo una P o tengo una prueba de que no hay P.
// Se usa escribiendo  lem[X]  y haciendo match sobre el Either que devuelve.
trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

// Doble negación: si "no es verdad que no P", entonces P.
// Se usa escribiendo  dn[X](nx => ...)  cuando la meta es X.
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

// Plantilla de los isomorfismos: tú implementas from y to; las leyes las usan los tests.
trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)   // ir y volver deja igual una A
    def law2(b: B): Boolean = equalB(from(to(b)), b)   // ir y volver deja igual una B

// Árbol binario: o vacío, o un nodo con hijo izquierdo, raíz e hijo derecho.
enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

// Atajos para escribir árboles en los tests. El orden de los argumentos imita el dibujo:
// left(hijo, raiz) pone el hijo a la izquierda; right(raiz, hijo) a la derecha.
object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)

import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)

import $ivy.$                                

import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.

defined type Not
defined type <=>
defined trait ExcludedMiddle
defined trait DoubleNegation
defined trait Isomorphic
defined class Tree
defined object Tree
import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)


## a) Contar valores

| Tipo | Cardinalidad |
|---|---|
| `Nothing` / `Unit` / `Boolean` | 0 / 1 / 2 |
| `Either[A, B]` | \|A\| **+** \|B\| |
| `(A, B)` | \|A\| **·** \|B\| |
| `Option[A]` | \|A\| + 1 |
| `A => B` | \|B\| **^** \|A\| (¡base = resultado!) |

Por qué: un `Either` es *o* una A *o* una B (se suman las posibilidades); un par es una A *y* una B (todas las combinaciones: se multiplican); una función `A => B` elige una B para **cada** A (|B| opciones, |A| veces: potencia).

Se escribe una igualdad por línea aplicando las reglas del álgebra:

```
|Option[A] => B| =
|B| ^ |Option[A]| =
|B| ^ (|A| + 1) =
|B| ^ |A| * |B| ^ 1 =
|A => B| * |B| =
|(A => B, B)|
```

Reglas que más salen: `a·(b+c) = ab+ac`, `a^(b+c) = a^b·a^c`, `(a·b)^c = a^c·b^c`, `a^1 = a`, conmutativa.

Si te dicen "**demuestra o refuta**" y las cuentas no salen iguales, no son isomorfos: lo justificas con un ejemplo (por ejemplo `A = Unit`).

## b) La receta de `from` / `to`

1. Haz una **tabla**: cada forma del tipo izquierdo ↔ su forma en el derecho.
2. `from` = la tabla de izquierda a derecha, con `match`. `to` = de derecha a izquierda.
3. Si la entrada es una **función** con dominio pequeño, evalúala en todos los valores: `f(false)`, `f(true)`; `f(None)`, `a => f(Some(a))`; `b => f(Left(b))`, `c => f(Right(c))`.
4. Si tienes que **devolver una función**, usa una lambda o un bloque de `case`.

## Ejemplo 1 · `Option[A] => B` ≅ `(B, A => B)`

In [ ]:
class Iso1[A, B] extends Isomorphic[Option[A] => B, (B, A => B)]:

    // from: me dan una FUNCIÓN f que acepta Option[A]. Un Option[A] solo puede ser
    // None o Some(a), así que f queda totalmente descrita por dos cosas:
    //   - lo que devuelve para None        → un valor B
    //   - lo que devuelve para cada Some(a) → una función A => B
    // Esas dos cosas son justo el par (B, A => B).
    def from(f: Option[A] => B): (B, A => B) =
        (f(None),             // evalúo f en None
         a => f(Some(a)))     // y "en todos los Some": función que, dada a, evalúa f(Some(a))

    // to: tengo que DEVOLVER una función Option[A] => B. La escribo con un case por
    // cada forma del Option, y uso el componente del par que corresponde a cada una.
    // Es la operación inversa de from: lo que from separó, to lo vuelve a juntar.
    def to(p: (B, A => B)): Option[A] => B =
        case None    => p._1        // para None, el valor guardado
        case Some(a) => p._2(a)     // para Some(a), aplicar la función guardada

## Ejemplo 2 · `(A, Either[B, C])` ≅ `Either[(A, B), (A, C)]`

Tabla: `(a, Left(b))` ↔ `Left((a, b))`; `(a, Right(c))` ↔ `Right((a, c))`.

In [ ]:
// Es la propiedad distributiva a·(b+c) = a·b + a·c: "una A y (una B o una C)"
// es lo mismo que "(una A y una B) o (una A y una C)".
class Iso2[A, B, C] extends Isomorphic[(A, Either[B, C]), Either[(A, B), (A, C)]]:

    // Hay tantos case como formas tiene la entrada (dos), y cada uno sale de la tabla.
    def from(t: (A, Either[B, C])): Either[(A, B), (A, C)] =
        t match
            case (a, Left(b))  => Left((a, b))    // el Either interior decide el lado exterior
            case (a, Right(c)) => Right((a, c))
            // Ojo a los dobles paréntesis: Left((a, b)) mete UN par dentro del Left.

    // La misma tabla leída al revés. Si cruzara dos casos (p. ej. Left → Right),
    // compilaría igual, pero las leyes fallarían: por eso conviene escribir la tabla antes.
    def to(e: Either[(A, B), (A, C)]): (A, Either[B, C]) =
        e match
            case Left((a, b))  => (a, Left(b))
            case Right((a, c)) => (a, Right(c))

In [ ]:
// Comprobación rápida de las leyes con tipos concretos (Int, Char, String):
// law1: to(from(x)) == x     law2: from(to(y)) == y.  Deben salir las dos true.
val iso = Iso2[Int, Char, String]()
(iso.law1((1, Left('a'))), iso.law2(Right((2, "hola"))))

## Mini-ejercicio (5 min)

`Either[B, C] => A` ≅ `(B => A, C => A)`. Primero la cadena, luego el código.

In [1]:
class IsoEj[A, B, C] extends Isomorphic[Either[B, C] => A, (B => A, C => A)]:

    def from(f: Either[B, C] => A): (B => A, C => A) =
        (Left(b) => a, Right(c) => a)

    def to(t: (B => A, C => A)): Either[B, C] => A =
        case Left(b) => a
        case Right(c) => a

(console)
-- Error: <splitter>:4:13 ------------------------------------------------------
4 |        (Left(b) => a, Right(c) => a)
  |         ^^^^^^^
  |         not a legal formal parameter for a function literal
-- Error: <splitter>:4:28 ------------------------------------------------------
4 |        (Left(b) => a, Right(c) => a)
  |                       ^^^^^^^^
  |                       not a legal formal parameter for a function literal

<details>
<summary><b>Ver solución</b></summary>

```
|Either[B, C] => A| = |A| ^ (|B| + |C|) = |A|^|B| * |A|^|C| = |B => A| * |C => A| = |(B => A, C => A)|
```

```scala
class IsoEj[A, B, C] extends Isomorphic[Either[B, C] => A, (B => A, C => A)]:

    // f acepta un Either: su comportamiento se divide en "qué hace con los Left"
    // y "qué hace con los Right". Cada parte es una función.
    def from(f: Either[B, C] => A): (B => A, C => A) =
        (b => f(Left(b)),      // la parte de los Left
         c => f(Right(c)))     // la parte de los Right

    // Para devolver una función sobre Either, un case por lado,
    // usando la función del par que corresponde.
    def to(t: (B => A, C => A)): Either[B, C] => A =
        case Left(b)  => t._1(b)
        case Right(c) => t._2(c)
```

</details>

**Siguiente:** `04-ejercicio3-listas.ipynb`